# Caso 1: Agente avanzado de pronóstico de flujo de efectivo corporativo

Objetivo: pronosticar flujo neto, detectar riesgo de liquidez y generar un dashboard ejecutivo avanzado con agente IA renderizado como chat.


## Nota de compatibilidad corregida
Este notebook evita el uso de `mean_squared_error(..., squared=False)` porque algunas versiones de `scikit-learn` no aceptan el argumento `squared`. En su lugar, calcula el RMSE como `sqrt(MSE)`.


In [ ]:
# Instalación mínima para Colab
# Si ya están instaladas, esta celda no causa problema.
!pip -q install pandas numpy scikit-learn plotly yfinance openpyxl

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report

# Opcional: montar Google Drive en Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Corporate_AI_Lab'
    os.makedirs(BASE_DIR, exist_ok=True)
except Exception:
    BASE_DIR = '.'

print('Carpeta de trabajo:', BASE_DIR)

# Función compatible para calcular RMSE en distintas versiones de scikit-learn
def calcular_rmse(y_real, y_pred):
    """Calcula RMSE sin depender del argumento squared=False.
    Algunas versiones de scikit-learn no aceptan squared en mean_squared_error.
    """
    return np.sqrt(mean_squared_error(y_real, y_pred))


In [ ]:

# Cargar base: intenta Drive; si no existe, usa archivo local cargado al entorno
import os, pandas as pd, numpy as np
from pathlib import Path

possible_paths = [
    os.path.join(BASE_DIR, 'flujo_efectivo_sintetico.csv'),
    '/content/flujo_efectivo_sintetico.csv',
    'flujo_efectivo_sintetico.csv'
]
path = next((p for p in possible_paths if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError('Sube flujo_efectivo_sintetico.csv o guárdalo en Corporate_AI_Lab dentro de Drive')

df = pd.read_csv(path)
df['mes'] = pd.to_datetime(df['mes'])
df = df.sort_values('mes')
df.head()


In [ ]:

# Ingeniería de variables para pronóstico de flujo neto mensual
df['mes_num'] = np.arange(len(df))
df['mes_calendario'] = df['mes'].dt.month
df['trimestre'] = df['mes'].dt.quarter
for lag in [1,2,3]:
    df[f'flujo_lag_{lag}'] = df['flujo_neto'].shift(lag)
    df[f'ventas_lag_{lag}'] = df['ventas'].shift(lag)

df_model = df.dropna().copy()
features = ['mes_num','mes_calendario','trimestre','ventas','porcentaje_cobranza','cobranza','costos_variables','gastos_fijos','nomina','impuestos','servicio_deuda','capex','flujo_lag_1','flujo_lag_2','flujo_lag_3','ventas_lag_1','ventas_lag_2','ventas_lag_3']
target = 'flujo_neto'
X = df_model[features]
y = df_model[target]

# División temporal: últimos 8 meses como prueba
train_size = len(df_model)-8
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

model = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = calcular_rmse(y_test, pred)
r2 = r2_score(y_test, pred)
print(f'MAE: ${mae:,.0f}')
print(f'RMSE: ${rmse:,.0f}')
print(f'R2: {r2:.3f}')

resultado = df_model.iloc[train_size:][['mes','flujo_neto','saldo_final']].copy()
resultado['flujo_predicho'] = pred
resultado


In [ ]:

# Pronóstico simple a 6 meses usando crecimiento base y rezagos actualizados
future_rows = []
last_df = df.copy()
last_date = last_df['mes'].max()

for i in range(1,7):
    new_date = last_date + pd.DateOffset(months=i)
    hist = last_df.tail(3)
    ventas_est = hist['ventas'].mean() * (1 + 0.012*i)
    porc_cob = min(0.92, max(0.68, hist['porcentaje_cobranza'].mean()))
    cobranza_est = ventas_est * porc_cob
    costos_est = ventas_est * hist['costos_variables'].sum()/hist['ventas'].sum()
    gastos_est = hist['gastos_fijos'].mean() * 1.006
    nomina_est = hist['nomina'].mean() * 1.004
    impuestos_est = max(0, (ventas_est-costos_est-gastos_est-nomina_est)*0.22)
    deuda_est = hist['servicio_deuda'].mean()
    capex_est = hist['capex'].median()
    tmp = {
        'mes': new_date, 'ventas': ventas_est, 'porcentaje_cobranza': porc_cob,
        'cobranza': cobranza_est, 'costos_variables': costos_est, 'gastos_fijos': gastos_est,
        'nomina': nomina_est, 'impuestos': impuestos_est, 'servicio_deuda': deuda_est,
        'capex': capex_est, 'mes_num': len(last_df), 'mes_calendario': new_date.month,
        'trimestre': ((new_date.month-1)//3)+1,
        'flujo_lag_1': last_df['flujo_neto'].iloc[-1], 'flujo_lag_2': last_df['flujo_neto'].iloc[-2], 'flujo_lag_3': last_df['flujo_neto'].iloc[-3],
        'ventas_lag_1': last_df['ventas'].iloc[-1], 'ventas_lag_2': last_df['ventas'].iloc[-2], 'ventas_lag_3': last_df['ventas'].iloc[-3]
    }
    flujo_pred = model.predict(pd.DataFrame([tmp])[features])[0]
    tmp['flujo_neto'] = flujo_pred
    tmp['saldo_final'] = last_df['saldo_final'].iloc[-1] + flujo_pred
    future_rows.append(tmp)
    append_cols = ['mes','ventas','porcentaje_cobranza','cobranza','costos_variables','gastos_fijos','nomina','impuestos','servicio_deuda','capex','flujo_neto','saldo_final']
    last_df = pd.concat([last_df, pd.DataFrame([{k:tmp[k] for k in append_cols}])], ignore_index=True)

forecast = pd.DataFrame(future_rows)
forecast[['mes','ventas','flujo_neto','saldo_final']]


In [ ]:

# Agente ejecutivo basado en reglas: interpreta el pronóstico y propone acciones
min_saldo = forecast['saldo_final'].min()
mes_critico = forecast.loc[forecast['saldo_final'].idxmin(),'mes'].strftime('%Y-%m')
flujo_prom = forecast['flujo_neto'].mean()
meses_neg = (forecast['flujo_neto'] < 0).sum()

if min_saldo < 500_000 or meses_neg >= 2:
    nivel = 'ALTO'
    accion = 'activar plan de liquidez: acelerar cobranza, diferir capex no crítico y renegociar pagos de corto plazo.'
elif min_saldo < 1_200_000 or meses_neg == 1:
    nivel = 'MEDIO'
    accion = 'monitorear semanalmente cobranza, controlar gasto variable y mantener reserva mínima.'
else:
    nivel = 'BAJO'
    accion = 'mantener operación y evaluar uso productivo del excedente sin comprometer liquidez.'

reporte = f"""
# Agente de Flujo de Efectivo\n\n**Nivel de riesgo de liquidez:** {nivel}\n\n**Mes más sensible:** {mes_critico}\n\n**Flujo promedio proyectado:** ${flujo_prom:,.0f}\n\n**Saldo mínimo proyectado:** ${min_saldo:,.0f}\n\n**Recomendación ejecutiva:** Se recomienda {accion}\n\n**Nota metodológica:** Modelo Random Forest entrenado con ventas, cobranza, costos, gastos, deuda, capex y rezagos de flujo. Este resultado es una estimación, no una garantía.\n"""
print(reporte)


In [ ]:
# Dashboard HTML avanzado estilo magIA / Bloomberg
# Incluye: KPIs, pestañas, visualizaciones Plotly, tabla ejecutiva y agente renderizado como chat.

import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display, IFrame

# -----------------------------
# 1) Preparación de datos
# -----------------------------
hist = df[['mes','ventas','cobranza','costos_variables','gastos_fijos','nomina','impuestos','servicio_deuda','capex','flujo_neto','saldo_final']].copy()
hist['tipo'] = 'Histórico'

fut = forecast[['mes','ventas','cobranza','costos_variables','gastos_fijos','nomina','impuestos','servicio_deuda','capex','flujo_neto','saldo_final']].copy()
fut['tipo'] = 'Pronóstico'

plot_df = pd.concat([hist, fut], ignore_index=True).sort_values('mes')
forecast_display = forecast.copy()
forecast_display['mes'] = forecast_display['mes'].dt.strftime('%Y-%m')

# Indicadores ejecutivos
ultimo_saldo_hist = float(df['saldo_final'].iloc[-1])
saldo_final_proy = float(forecast['saldo_final'].iloc[-1])
cambio_saldo = saldo_final_proy - ultimo_saldo_hist
flujo_total_proy = float(forecast['flujo_neto'].sum())
meses_neg = int((forecast['flujo_neto'] < 0).sum())
min_saldo = float(forecast['saldo_final'].min())
max_saldo = float(forecast['saldo_final'].max())
mes_critico = forecast.loc[forecast['saldo_final'].idxmin(), 'mes'].strftime('%Y-%m')

# Colores por nivel de riesgo
risk_palette = {
    'BAJO': {'main':'#35e6a7', 'soft':'rgba(53,230,167,.16)', 'label':'Liquidez controlada'},
    'MEDIO': {'main':'#ffbf5f', 'soft':'rgba(255,191,95,.17)', 'label':'Monitoreo requerido'},
    'ALTO': {'main':'#ff5b7f', 'soft':'rgba(255,91,127,.18)', 'label':'Acción prioritaria'}
}
risk = risk_palette.get(nivel, risk_palette['MEDIO'])

def money(x):
    return f"${x:,.0f}"

def pct(x):
    try:
        return f"{x:.1%}"
    except Exception:
        return "N/D"

# -----------------------------
# 2) Figuras Plotly avanzadas
# -----------------------------
plot_template = "plotly_dark"

fig_cash = go.Figure()
for tipo, color in [('Histórico', '#31d7ff'), ('Pronóstico', '#ffbf5f')]:
    d = plot_df[plot_df['tipo'] == tipo]
    fig_cash.add_trace(go.Scatter(
        x=d['mes'], y=d['flujo_neto'],
        mode='lines+markers',
        name=tipo,
        line=dict(width=3, color=color),
        marker=dict(size=7),
        hovertemplate='<b>%{x|%Y-%m}</b><br>Flujo neto: $%{y:,.0f}<extra></extra>'
    ))
fig_cash.add_hline(y=0, line_dash='dot', line_color='#9aa7b8', annotation_text='Punto de equilibrio')
fig_cash.update_layout(
    template=plot_template,
    title='Flujo neto histórico y proyectado',
    height=430,
    margin=dict(l=30, r=20, t=55, b=30),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(6,16,30,.72)',
    legend=dict(orientation='h', y=1.12, x=.02),
    hovermode='x unified'
)

fig_balance = go.Figure()
fig_balance.add_trace(go.Scatter(
    x=plot_df['mes'], y=plot_df['saldo_final'],
    mode='lines',
    name='Saldo final',
    fill='tozeroy',
    line=dict(width=3, color='#35e6a7'),
    hovertemplate='<b>%{x|%Y-%m}</b><br>Saldo final: $%{y:,.0f}<extra></extra>'
))
fig_balance.add_trace(go.Scatter(
    x=forecast['mes'], y=forecast['saldo_final'],
    mode='markers',
    name='Meses proyectados',
    marker=dict(size=10, color='#ffbf5f', line=dict(width=1, color='#ffffff')),
    hovertemplate='<b>%{x|%Y-%m}</b><br>Saldo proyectado: $%{y:,.0f}<extra></extra>'
))
fig_balance.add_hline(y=500000, line_dash='dash', line_color='#ff5b7f', annotation_text='Zona crítica')
fig_balance.add_hline(y=1200000, line_dash='dot', line_color='#ffbf5f', annotation_text='Reserva mínima')
fig_balance.update_layout(
    template=plot_template,
    title='Saldo final y bandas de liquidez',
    height=430,
    margin=dict(l=30, r=20, t=55, b=30),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(6,16,30,.72)',
    hovermode='x unified'
)

components = forecast[['mes','cobranza','costos_variables','gastos_fijos','nomina','impuestos','servicio_deuda','capex']].copy()
components['costos_variables'] *= -1
components['gastos_fijos'] *= -1
components['nomina'] *= -1
components['impuestos'] *= -1
components['servicio_deuda'] *= -1
components['capex'] *= -1
comp_long = components.melt(id_vars='mes', var_name='concepto', value_name='monto')
label_map = {
    'cobranza':'Cobranza',
    'costos_variables':'Costos variables',
    'gastos_fijos':'Gastos fijos',
    'nomina':'Nómina',
    'impuestos':'Impuestos',
    'servicio_deuda':'Servicio deuda',
    'capex':'Capex'
}
comp_long['concepto'] = comp_long['concepto'].map(label_map)
fig_components = px.bar(
    comp_long, x='mes', y='monto', color='concepto',
    title='Drivers del flujo proyectado: entradas y salidas',
    color_discrete_sequence=['#35e6a7','#ff5b7f','#ff8a3d','#b56dff','#31d7ff','#ffbf5f','#e75cff']
)
fig_components.update_layout(
    template=plot_template, height=430,
    margin=dict(l=30, r=20, t=55, b=30),
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(6,16,30,.72)',
    legend=dict(orientation='h', y=-.22, x=.02),
    hovermode='x unified'
)

fig_model = go.Figure()
fig_model.add_trace(go.Scatter(
    x=resultado['mes'], y=resultado['flujo_neto'],
    mode='lines+markers', name='Real',
    line=dict(color='#31d7ff', width=3)
))
fig_model.add_trace(go.Scatter(
    x=resultado['mes'], y=resultado['flujo_predicho'],
    mode='lines+markers', name='Predicho',
    line=dict(color='#ffbf5f', width=3, dash='dash')
))
fig_model.update_layout(
    template=plot_template,
    title='Backtest: flujo real vs flujo predicho',
    height=390,
    margin=dict(l=30, r=20, t=55, b=30),
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(6,16,30,.72)',
    hovermode='x unified'
)

imp = pd.DataFrame({
    'variable': features,
    'importancia': model.feature_importances_
}).sort_values('importancia', ascending=True).tail(10)
fig_importance = px.bar(
    imp, x='importancia', y='variable', orientation='h',
    title='Top 10 variables que explican el modelo',
    color='importancia',
    color_continuous_scale=['#31d7ff', '#b56dff', '#ffbf5f']
)
fig_importance.update_layout(
    template=plot_template,
    height=390,
    margin=dict(l=30, r=20, t=55, b=30),
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(6,16,30,.72)',
    coloraxis_showscale=False
)

# Tablas formateadas
forecast_table = forecast[['mes','ventas','cobranza','costos_variables','gastos_fijos','nomina','impuestos','servicio_deuda','capex','flujo_neto','saldo_final']].copy()
forecast_table['mes'] = forecast_table['mes'].dt.strftime('%Y-%m')
for col in forecast_table.columns:
    if col != 'mes':
        forecast_table[col] = forecast_table[col].map(lambda v: f"${v:,.0f}")
forecast_html = forecast_table.to_html(index=False, classes='data-table', border=0)

resultado_table = resultado.copy()
resultado_table['mes'] = resultado_table['mes'].dt.strftime('%Y-%m')
for col in ['flujo_neto','saldo_final','flujo_predicho']:
    resultado_table[col] = resultado_table[col].map(lambda v: f"${v:,.0f}")
resultado_html = resultado_table.to_html(index=False, classes='data-table', border=0)

# Convertir figuras a HTML parcial
fig_cash_html = fig_cash.to_html(full_html=False, include_plotlyjs='cdn', config={'displayModeBar': True, 'responsive': True})
fig_balance_html = fig_balance.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': True, 'responsive': True})
fig_components_html = fig_components.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': True, 'responsive': True})
fig_model_html = fig_model.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': True, 'responsive': True})
fig_importance_html = fig_importance.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': True, 'responsive': True})

# Mensajes del agente
agent_bullets = [
    f"El riesgo de liquidez se clasifica como {nivel}: {risk['label']}.",
    f"El mes más sensible es {mes_critico}, con saldo mínimo proyectado de {money(min_saldo)}.",
    f"El flujo neto acumulado de los próximos 6 meses es {money(flujo_total_proy)}.",
    f"Se detectan {meses_neg} meses con flujo neto negativo.",
    f"Acción sugerida: {accion.capitalize()}"
]
agent_html = "".join([f"<div class='msg assistant'><div class='avatar'>IA</div><div class='bubble'>{b}</div></div>" for b in agent_bullets])

# -----------------------------
# 3) HTML final
# -----------------------------
html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>magIA · Agente de Flujo de Efectivo</title>
<style>
:root {{
    --bg: #050b14;
    --panel: rgba(10, 23, 40, .92);
    --panel2: rgba(12, 30, 54, .88);
    --line: rgba(49,215,255,.22);
    --text: #eaf6ff;
    --muted: #9fb7cf;
    --cyan: #31d7ff;
    --green: #35e6a7;
    --amber: #ffbf5f;
    --pink: #ff5b7f;
    --violet: #b56dff;
    --orange: #ff8a3d;
    --risk: {risk['main']};
    --risk-soft: {risk['soft']};
}}
* {{ box-sizing: border-box; }}
body {{
    margin: 0;
    font-family: Inter, Segoe UI, Roboto, Arial, sans-serif;
    background:
        radial-gradient(circle at 12% 8%, rgba(49,215,255,.18), transparent 28%),
        radial-gradient(circle at 82% 4%, rgba(181,109,255,.16), transparent 25%),
        linear-gradient(135deg, #040812 0%, #071526 50%, #0a1020 100%);
    color: var(--text);
}}
.container {{ width: min(1420px, calc(100% - 34px)); margin: 0 auto; }}
.header {{
    padding: 28px 0 20px;
    border-bottom: 1px solid var(--line);
    background: rgba(4,10,18,.66);
    backdrop-filter: blur(14px);
    position: sticky;
    top: 0;
    z-index: 20;
}}
.brand {{
    display:flex; align-items:center; justify-content:space-between; gap:20px;
}}
.logo {{
    display:flex; align-items:center; gap:14px;
}}
.logo-mark {{
    width:54px; height:54px; border-radius:18px;
    background: linear-gradient(135deg, var(--cyan), var(--violet));
    display:grid; place-items:center;
    color:#00121f; font-weight:900; font-size:22px;
    box-shadow: 0 0 32px rgba(49,215,255,.35);
}}
.kicker {{ color: var(--cyan); letter-spacing:.12em; text-transform:uppercase; font-weight:800; font-size:12px; }}
h1 {{ margin: 4px 0 0; font-size: clamp(28px, 4vw, 54px); line-height: 1.02; }}
.subtitle {{ color: var(--muted); margin-top: 8px; font-size: 16px; max-width: 880px; line-height: 1.6; }}
.status-pill {{
    border:1px solid var(--risk); background: var(--risk-soft);
    color: var(--risk); padding: 12px 16px; border-radius:999px;
    font-weight:900; display:inline-flex; align-items:center; gap:10px;
}}
.status-dot {{
    width:10px; height:10px; border-radius:50%; background: var(--risk);
    box-shadow: 0 0 18px var(--risk);
}}
.kpi-grid {{
    display:grid; grid-template-columns: repeat(5, minmax(170px,1fr)); gap:14px;
    margin: 24px 0;
}}
.kpi {{
    background: linear-gradient(180deg, rgba(12,30,54,.88), rgba(7,18,32,.92));
    border:1px solid rgba(255,255,255,.07);
    border-radius: 22px;
    padding:18px;
    box-shadow: 0 18px 45px rgba(0,0,0,.25);
    position:relative; overflow:hidden;
}}
.kpi:before {{
    content:""; position:absolute; inset:0 0 auto 0; height:3px;
    background: linear-gradient(90deg, var(--cyan), var(--violet), var(--amber));
}}
.kpi .label {{ color: var(--muted); font-size:13px; margin-bottom:8px; }}
.kpi .value {{ font-size: clamp(18px, 2vw, 28px); font-weight:900; }}
.kpi .hint {{ color: var(--muted); font-size:12px; margin-top:8px; }}
.tabs {{
    display:flex; gap:10px; flex-wrap:wrap; margin: 12px 0 18px;
}}
.tab-btn {{
    background: rgba(255,255,255,.04);
    border:1px solid rgba(255,255,255,.09);
    color: var(--muted);
    padding: 12px 15px;
    border-radius: 999px;
    cursor:pointer;
    font-weight:800;
}}
.tab-btn.active {{
    color:#00121f;
    background: linear-gradient(135deg, var(--cyan), #8ff2ff);
    border-color: transparent;
}}
.tab {{ display:none; }}
.tab.active {{ display:block; }}
.grid-2 {{ display:grid; grid-template-columns: 1.1fr .9fr; gap:18px; }}
.grid-3 {{ display:grid; grid-template-columns: repeat(3, 1fr); gap:18px; }}
.card {{
    background: var(--panel);
    border:1px solid var(--line);
    border-radius: 24px;
    padding: 18px;
    box-shadow: 0 18px 54px rgba(0,0,0,.28);
}}
.card h2, .card h3 {{ margin-top:0; }}
.chart-card {{ padding: 12px; }}
.agent-shell {{
    display:grid;
    grid-template-columns: 330px 1fr;
    gap:18px;
}}
.agent-profile {{
    background: linear-gradient(180deg, rgba(49,215,255,.10), rgba(181,109,255,.09));
    border:1px solid rgba(49,215,255,.18);
    border-radius:24px;
    padding:22px;
}}
.agent-orb {{
    width:96px; height:96px; border-radius:32px;
    background:
      radial-gradient(circle at 30% 25%, #ffffff 0%, #92f2ff 12%, transparent 28%),
      linear-gradient(135deg, var(--cyan), var(--violet));
    box-shadow: 0 0 45px rgba(49,215,255,.35);
    margin-bottom:18px;
    display:grid; place-items:center;
    color:#00111f; font-size:30px; font-weight:900;
}}
.chat {{
    background: rgba(2,9,18,.52);
    border: 1px solid rgba(255,255,255,.08);
    border-radius:24px;
    padding:18px;
    max-height: 620px;
    overflow:auto;
}}
.msg {{
    display:flex; gap:12px; margin: 14px 0;
    align-items:flex-start;
}}
.msg.user {{ justify-content:flex-end; }}
.avatar {{
    min-width:42px; height:42px; border-radius:15px;
    background: linear-gradient(135deg, var(--cyan), var(--violet));
    color:#00111f; display:grid; place-items:center;
    font-weight:900;
}}
.bubble {{
    background: rgba(255,255,255,.06);
    border:1px solid rgba(255,255,255,.09);
    padding:14px 16px;
    border-radius: 18px;
    line-height:1.55;
    color: #dbefff;
}}
.msg.user .bubble {{
    background: rgba(49,215,255,.12);
    border-color: rgba(49,215,255,.26);
}}
.action-list {{ display:grid; gap:12px; margin-top:16px; }}
.action {{
    border-left:4px solid var(--risk);
    background: rgba(255,255,255,.04);
    padding: 12px 14px;
    border-radius: 14px;
    color: var(--muted);
}}
.table-wrap {{
    overflow:auto;
    border:1px solid rgba(255,255,255,.08);
    border-radius:18px;
}}
.data-table {{
    border-collapse: collapse;
    width:100%;
    min-width:920px;
    font-size:13px;
}}
.data-table th {{
    background: rgba(49,215,255,.11);
    color:#bff4ff;
    text-align:left;
    padding:12px;
    position:sticky;
    top:0;
}}
.data-table td {{
    padding:10px 12px;
    border-bottom:1px solid rgba(255,255,255,.06);
    color:#d7e8f7;
}}
.badge {{
    display:inline-flex; padding:6px 10px; border-radius:999px;
    background: rgba(49,215,255,.12); color: var(--cyan);
    border:1px solid rgba(49,215,255,.2); font-size:12px; font-weight:900;
}}
.footer {{
    color: var(--muted); text-align:center; padding:30px 0 50px;
}}
@media(max-width: 1100px) {{
    .kpi-grid {{ grid-template-columns: repeat(2, 1fr); }}
    .grid-2, .grid-3, .agent-shell {{ grid-template-columns:1fr; }}
}}
@media(max-width: 700px) {{
    .kpi-grid {{ grid-template-columns: 1fr; }}
    .brand {{ align-items:flex-start; flex-direction:column; }}
}}
</style>
</head>
<body>
<header class="header">
  <div class="container brand">
    <div class="logo">
      <div class="logo-mark">mIA</div>
      <div>
        <div class="kicker">magIA · Corporate Finance AI Agent</div>
        <h1>Agente de Flujo de Efectivo</h1>
        <div class="subtitle">Dashboard ejecutivo de pronóstico, riesgo de liquidez, explicación del modelo y recomendaciones accionables con renderización tipo asistente IA.</div>
      </div>
    </div>
    <div class="status-pill"><span class="status-dot"></span> Riesgo {nivel}</div>
  </div>
</header>

<main class="container">
  <section class="kpi-grid">
    <div class="kpi"><div class="label">Saldo actual</div><div class="value">{money(ultimo_saldo_hist)}</div><div class="hint">Último mes histórico</div></div>
    <div class="kpi"><div class="label">Saldo proyectado final</div><div class="value">{money(saldo_final_proy)}</div><div class="hint">Cierre del horizonte</div></div>
    <div class="kpi"><div class="label">Cambio de saldo</div><div class="value">{money(cambio_saldo)}</div><div class="hint">Final proyectado vs actual</div></div>
    <div class="kpi"><div class="label">Flujo acumulado</div><div class="value">{money(flujo_total_proy)}</div><div class="hint">Suma próximos 6 meses</div></div>
    <div class="kpi"><div class="label">R² del modelo</div><div class="value">{r2:.3f}</div><div class="hint">Backtest temporal</div></div>
  </section>

  <nav class="tabs">
    <button class="tab-btn active" onclick="openTab(event,'tab-resumen')">Resumen ejecutivo</button>
    <button class="tab-btn" onclick="openTab(event,'tab-forecast')">Forecast financiero</button>
    <button class="tab-btn" onclick="openTab(event,'tab-modelo')">Modelo ML</button>
    <button class="tab-btn" onclick="openTab(event,'tab-agente')">Agente IA</button>
    <button class="tab-btn" onclick="openTab(event,'tab-datos')">Datos</button>
  </nav>

  <section id="tab-resumen" class="tab active">
    <div class="grid-2">
      <div class="card chart-card">{fig_cash_html}</div>
      <div class="card chart-card">{fig_balance_html}</div>
    </div>
    <div class="grid-3" style="margin-top:18px;">
      <div class="card">
        <span class="badge">Interpretación</span>
        <h3>Lectura ejecutiva</h3>
        <p>El modelo estima el comportamiento de caja a partir de ventas, cobranza, costos, gastos, deuda, capex y rezagos financieros.</p>
        <p><strong>Mes crítico:</strong> {mes_critico}<br><strong>Saldo mínimo:</strong> {money(min_saldo)}<br><strong>Meses negativos:</strong> {meses_neg}</p>
      </div>
      <div class="card">
        <span class="badge">Acción</span>
        <h3>Decisión sugerida</h3>
        <div class="action">{accion.capitalize()}</div>
      </div>
      <div class="card">
        <span class="badge">Gobierno del modelo</span>
        <h3>Uso responsable</h3>
        <p>El resultado debe usarse como soporte de decisión. Es recomendable validar supuestos de cobranza, gasto, capex y deuda antes de tomar decisiones definitivas.</p>
      </div>
    </div>
  </section>

  <section id="tab-forecast" class="tab">
    <div class="grid-2">
      <div class="card chart-card">{fig_components_html}</div>
      <div class="card">
        <h3>Tabla ejecutiva del pronóstico</h3>
        <div class="table-wrap">{forecast_html}</div>
      </div>
    </div>
  </section>

  <section id="tab-modelo" class="tab">
    <div class="grid-2">
      <div class="card chart-card">{fig_model_html}</div>
      <div class="card chart-card">{fig_importance_html}</div>
    </div>
    <div class="grid-3" style="margin-top:18px;">
      <div class="card"><h3>MAE</h3><div class="kpi"><div class="value">{money(mae)}</div><div class="hint">Error absoluto promedio.</div></div></div>
      <div class="card"><h3>RMSE</h3><div class="kpi"><div class="value">{money(rmse)}</div><div class="hint">Penaliza más errores grandes.</div></div></div>
      <div class="card"><h3>R²</h3><div class="kpi"><div class="value">{r2:.3f}</div><div class="hint">Capacidad explicativa en prueba.</div></div></div>
    </div>
    <div class="card" style="margin-top:18px;">
      <h3>Backtest detallado</h3>
      <div class="table-wrap">{resultado_html}</div>
    </div>
  </section>

  <section id="tab-agente" class="tab">
    <div class="agent-shell">
      <div class="agent-profile">
        <div class="agent-orb">IA</div>
        <h2>Agente financiero</h2>
        <p>Interpreta el pronóstico, clasifica el riesgo de liquidez y propone acciones concretas para tesorería y dirección financiera.</p>
        <div class="action-list">
          <div class="action"><strong>Riesgo:</strong> {nivel}</div>
          <div class="action"><strong>Prioridad:</strong> {risk['label']}</div>
          <div class="action"><strong>Mes crítico:</strong> {mes_critico}</div>
        </div>
      </div>
      <div class="chat">
        <div class="msg user"><div class="bubble">Analiza el pronóstico de flujo de efectivo y dame una recomendación ejecutiva.</div></div>
        {agent_html}
        <div class="msg assistant"><div class="avatar">IA</div><div class="bubble"><strong>Nota:</strong> este análisis combina reglas financieras y salida del modelo. Antes de ejecutarlo operativamente, conviene validar supuestos de cobranza, capex y deuda con el equipo responsable.</div></div>
      </div>
    </div>
  </section>

  <section id="tab-datos" class="tab">
    <div class="card">
      <h3>Base histórica usada por el modelo</h3>
      <p style="color:var(--muted)">Muestra de los últimos registros históricos cargados antes del pronóstico.</p>
      <div class="table-wrap">{df.tail(12).drop(columns=[c for c in ['mes_num','mes_calendario','trimestre','flujo_lag_1','flujo_lag_2','flujo_lag_3','ventas_lag_1','ventas_lag_2','ventas_lag_3'] if c in df.columns]).to_html(index=False, classes='data-table', border=0)}</div>
    </div>
  </section>
</main>

<footer class="footer">
  magIA · Dashboard generado con Python, Machine Learning y Plotly · Dra. Elda C. Morales
</footer>

<script>
function openTab(evt, id) {{
  var i, tabcontent, tablinks;
  tabcontent = document.getElementsByClassName("tab");
  for (i = 0; i < tabcontent.length; i++) {{
    tabcontent[i].classList.remove("active");
  }}
  tablinks = document.getElementsByClassName("tab-btn");
  for (i = 0; i < tablinks.length; i++) {{
    tablinks[i].classList.remove("active");
  }}
  document.getElementById(id).classList.add("active");
  evt.currentTarget.classList.add("active");
  setTimeout(function() {{
    window.dispatchEvent(new Event('resize'));
  }}, 120);
}}
</script>
</body>
</html>
"""

out = os.path.join(BASE_DIR, 'dashboard_flujo_efectivo_AVANZADO.html')
with open(out, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"Dashboard avanzado guardado en: {out}")
display(HTML(f"""
<div style="font-family:Arial; padding:18px; border-radius:14px; background:#071526; color:#eaf6ff; border:1px solid #31d7ff;">
  <h3 style="margin-top:0;">Dashboard avanzado generado correctamente</h3>
  <p>Archivo creado: <strong>{out}</strong></p>
  <p>Ábrelo desde el panel de archivos de Colab o descárgalo para visualizarlo en navegador.</p>
</div>
"""))

try:
    display(IFrame(out, width='100%', height=760))
except Exception:
    pass
